# P3: Assemble the Clean Training Dataset (Provenance)

Merges four research notebooks (*Training Images from Seismic Events*,
*Image List Generator*, *Copy Dataset and Offset Images*, *Delete Consecutive
Images*) into the sequential workflow that built the Stage-1 "clean"
dataset. Requires the RGB archive; every stage ends in a manual review
whose final products already ship with this repository: the curated
`Dataset_2014_2015` folders are the end state of this entire pipeline plus
the denoising in P4.

Workflow: **3.1** harvest candidate frames around each seismically
detected large eruption, then *manually trim to 5 frames per event*;
**3.2** turn the curated folders into CSV file lists; **3.3** copy offset
frames to build the no-eruption class, then *manually verify them*;
**3.4** collapse back-to-back small eruptions so 3-frame windows never
collide.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "config.py").exists():
    if _root.parent == _root:
        raise FileNotFoundError(
            "config.py not found. Launch Jupyter from inside the repository, "
            "or place this notebook back in its original folder.")
    _root = _root.parent
sys.path.insert(0, str(_root))

import config

# This notebook reads the temporal-RGB archive; the monochrome archive is not
# used here (P2 consumes that one). Press Enter to accept the config.py value
# shown in brackets, or paste the archive folder path. Surrounding quotes and
# a leading file:// prefix are stripped automatically. A web link will not
# work: the archive must already be downloaded and unzipped on this machine.

def ask_path(label, default, must_exist):
    raw = input(f"{label}\n  [{default}]: ").strip().strip('"').strip("'")
    if raw.lower().startswith(("http://", "https://")):
        raise ValueError(
            "That is a web link. Download and unzip the archive first, then "
            "re-run this cell and give the folder path on this machine.")
    if raw.lower().startswith("file://"):
        raw = raw[7:]
    path = (Path(raw).expanduser() if raw else Path(default)).resolve()
    if must_exist and not path.is_dir():
        raise FileNotFoundError(
            f"No folder found at {path}. Check the path and re-run this cell.")
    return path

config.RGB_ARCHIVE_DIR = ask_path(
    "Temporal-RGB archive folder (the output of P2)",
    config.RGB_ARCHIVE_DIR, must_exist=True)

config.describe()

In [ ]:
import os, csv, shutil
from datetime import datetime, timedelta
import pandas as pd

RGB_ROOT = config.RGB_ARCHIVE_DIR
STAGING = config.OUTPUT_DIR / "dataset_staging"
STAGING.mkdir(exist_ok=True)


## 3.1 Harvest candidate frames around seismic events

For each seed time in `MasterList.txt`, copy every RGB frame in a ±20 s
window into a numbered folder for review. The window is deliberately
generous; the published catalog keeps exactly the 5 frames spanning each
eruption.

> **Manual step (output already provided).** The trimmed 5-frame events are the
> `Large Eruptions` folder of the shipped dataset.

In [ ]:
with open(config.MASTERLIST_PATH) as f:
    events = [line.strip() for line in f if line.strip()]

CAMERA_LEAD_S = 20    # start of harvest window relative to seismic time
WINDOW_S = 40         # total harvest window length

def hour_dir_of(dt):
    return RGB_ROOT / f"{dt:%Y}" / f"{dt:%m}" / f"{dt:%d}" / f"{dt:%H}"

for count, event in enumerate(events, start=1):
    t0 = datetime.strptime(event[:19], "%Y-%m-%dT%H:%M:%S") - timedelta(seconds=CAMERA_LEAD_S)
    wanted = {(t0 + timedelta(seconds=s)).strftime("%Y-%m-%d_%H%M%S") for s in range(WINDOW_S)}
    hour_dirs = {hour_dir_of(t0), hour_dir_of(t0 + timedelta(seconds=WINDOW_S - 1))}
    hour_dirs = [d for d in sorted(hour_dirs) if d.exists()]
    if not hour_dirs:
        print(f"[{count}] no archive hour folders for {event}")
        continue
    dest = STAGING / "large_candidates" / str(count)
    dest.mkdir(parents=True, exist_ok=True)
    n = 0
    for hour_dir in hour_dirs:
        for image in sorted(os.listdir(hour_dir)):
            if any(image.startswith(w) for w in wanted):
                shutil.copyfile(hour_dir / image, dest / image)
                n += 1
    print(f"[{count}] {event}: {n} candidate frames")


## 3.2 Build CSV file lists from curated folders

After manual curation, each class folder becomes a flat CSV of filenames.
These lists drive the offset copier below. (`file_list_small.csv` came from
the Honarbakhsh & Morra (2023) small-eruption catalog.)

In [ ]:
def folder_to_csv(folder, out_csv):
    names = []
    for dirpath, dirnames, filenames in os.walk(folder):
        dirnames.sort()
        names.extend(sorted(filenames))
    with open(out_csv, "w") as f:
        f.write("\n".join(names))
    print(f"{len(names)} filenames -> {out_csv}")

# Example:
# folder_to_csv(STAGING / "large_curated", STAGING / "file_list_large.csv")
# folder_to_csv(STAGING / "small_curated", STAGING / "file_list_small.csv")


## 3.3 Copy time-offset frames (builds the no-eruption class)

For each event frame, copy the frame a fixed number of *positions* away in
the time-sorted archive (one position is about 2 s). Offsets used for the
published dataset: note some no-eruption samples deliberately come from
*after* eruptions, teaching the network that post-eruptive cooling is not an
eruption:

| purpose | source list | offsets (frames) | about  time |
|---|---|---|---|
| Large-eruption frames | large list | -2 to +2 | ±4 s |
| Small-eruption frames | singles list | -2 to 0 | -4 s to 0 |
| No-eruption (near large) | large list | -900, +15, +150, +300, +900 | -30 m, +30 s, +5 m, +10 m, +30 m |
| No-eruption (near small) | singles list | -900, +900 | ±30 m |

The copier walks into adjacent hour folders when an offset crosses a folder
boundary, giving up outside Dec 2013 to Jan 2016.

> **Manual step (output already provided).** Copied frames were manually verified
> before entering the class folders.

In [ ]:
def hour_dir_for(dt):
    return RGB_ROOT / f"{dt:%Y}" / f"{dt:%m}" / f"{dt:%d}" / f"{dt:%H}"

def copy_offset_frames(csv_file, offsets, dest_folder):
    dest_folder.mkdir(parents=True, exist_ok=True)
    copied = missing = 0
    with open(csv_file, newline="") as f:
        for row in csv.reader(f):
            if not row:
                continue
            name = row[0].replace(".png", ".jpg")
            try:
                anchor_dt = datetime.strptime(name[:17], "%Y-%m-%d_%H%M%S")
            except ValueError:
                continue
            for offset in offsets:
                target, remaining, probe = None, offset, anchor_dt
                # Walk hour folders until the offset is absorbed
                for _hop in range(4000):
                    d = hour_dir_for(probe)
                    listing = sorted(os.listdir(d)) if d.exists() else []
                    if d.exists() and _hop == 0:
                        try:
                            base = listing.index(name)
                        except ValueError:
                            break
                        k = base + remaining
                        if 0 <= k < len(listing):
                            target = d / listing[k]; break
                        remaining = k if k < 0 else k - len(listing)
                    elif d.exists():
                        if remaining < 0 and abs(remaining) <= len(listing):
                            target = d / listing[remaining]; break
                        if remaining >= 0 and remaining < len(listing):
                            target = d / listing[remaining]; break
                        remaining += len(listing) if remaining < 0 else -len(listing)
                    probe += timedelta(hours=1 if offset > 0 else -1)
                    if probe.year < 2013 or probe.year > 2016:
                        break
                if target and target.exists():
                    out = dest_folder / target.name
                    if not out.exists():
                        shutil.copy2(target, out); copied += 1
                else:
                    missing += 1
    print(f"{csv_file}: copied {copied}, unresolved {missing}")

# Example (uncomment with your curated lists):
# copy_offset_frames(STAGING / "file_list_large.csv", [-900, 15, 150, 300, 900],
#                    STAGING / "no_eruption_candidates")


## 3.4 Collapse consecutive small eruptions

Two small eruptions less than 6 s apart would have overlapping 3-frame
windows, breaking event separation during the split. Keep only the first of
any such run (`file_list_small.csv` becomes `file_list_small_singles.csv`); use
the *full* list when deciding which no-eruption frames to exclude, and the
*singles* list when building the dataset.

In [ ]:
def collapse_consecutive(list_path, save_path, window_s=6):
    dfl = pd.read_csv(list_path, header=None, names=["filename"])
    dfl["dt"] = dfl["filename"].map(
        lambda n: datetime.strptime(n.split("_")[0] + " " + n.split("_")[1][:6], "%Y-%m-%d %H%M%S"))
    dfl = dfl.sort_values("dt")
    kept, prev = [], None
    for _, row in dfl.iterrows():
        if prev is None or (row["dt"] - prev).total_seconds() > window_s:
            kept.append(row["filename"]); prev = row["dt"]
    pd.DataFrame(kept, columns=["filename"]).to_csv(save_path, index=False)
    print(f"{len(dfl)} -> {len(kept)} after collapsing (<= {window_s} s) -> {save_path}")

# collapse_consecutive(STAGING / "file_list_small.csv", STAGING / "file_list_small_singles.csv")


---
The clean dataset then went through the Stage-2 *expansion*: the Stage-1
model was run over all of December 2014 with R4, and every apparent false
positive (about 43,300 frames at threshold 0.5) was added to the
no-eruption class. Stage 3 is the denoising in **P4**.